[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/diffphore_smoke_test.ipynb)

# DiffPhore smoke test

**Sandbox notebook - not workshop material.**

The only question this notebook answers is: *does DiffPhore actually run in Colab today?*
It runs the repository's own worked example and compares the result with the reference
output that ships with the repository.

DiffPhore is a diffusion model for 3D ligand-pharmacophore mapping
([Yu et al., Nat Commun 2025](https://doi.org/10.1038/s41467-025-57485-3), MIT licence).
Nothing here is tied to any workshop project.

## What you will do

- Check what Python, PyTorch and GPU this Colab runtime has
- Clone DiffPhore and make its AncPhore binary executable
- Install the graph dependencies and see whether they import
- Run the worked example from the README
- Compare the output with the reference results in the repository

## 1. Inspect the runtime

DiffPhore was written against Python 3.9 and PyTorch 1.12 with CUDA 11.6. Colab has moved
on a long way since, and that gap is the most likely cause of failure. Record what we
actually have before installing anything.

A GPU is optional. `inference.py` falls back to CPU when CUDA is unavailable, so the test
still works on a CPU runtime, only slower.

In [ ]:
import sys, subprocess
print("Python", sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch not installed")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "No GPU")

## 2. Get DiffPhore

The model weights (about 9 MB) are committed in the repository, so there is no separate
checkpoint to download.

`AncPhore` is the pharmacophore engine DiffPhore calls out to. It is a Linux x86-64 binary
and arrives without the executable bit set, so we add it. It also needs the `programs/data`
folder that sits beside it.

In [ ]:
import os
if not os.path.exists("/content/DiffPhore"):
    !git clone -q https://github.com/VicFisher/DiffPhore.git /content/DiffPhore
!chmod +x /content/DiffPhore/programs/AncPhore
!ls -lh /content/DiffPhore/weights/diffphore_calibrated_warmuped_ft/
!/content/DiffPhore/programs/AncPhore --help 2>&1 | head -5

## 3. Install the graph dependencies

Do **not** build the conda environment the README describes. Its wheels are pinned to
`+pt112cu116`, which will not resolve against the PyTorch already in this runtime.

Install `torch-geometric` against Colab's existing PyTorch instead. The companion packages
(`torch-scatter`, `torch-sparse`, `torch-cluster`) are only needed if an import fails, and
they are slow to compile, so we leave them out until something asks for them.

In [ ]:
!pip install -q torch-geometric
import torch_geometric
print("torch_geometric", torch_geometric.__version__)

> **Note:** DiffPhore was written for torch-geometric 2.1. If an import fails here, pin it
> with `!pip install -q torch-geometric==2.1.0` and try again, then report the error.

## 4. Run the worked example

This is the single-pair command from the README, with two changes.

`--ancphore_path` defaults to the relative path `../programs/`, which only resolves when
run from inside `src/`, so we pass it in full. And `--sample_per_complex` drops from 40 to
4: we only care whether the thing runs, not whether the poses are good.

In [ ]:
!cd /content/DiffPhore && python src/inference.py \
  --phore   examples/phore/sQC_QFA_complex.phore \
  --ligand  examples/ligands/STK936575.sdf \
  --model_dir weights/diffphore_calibrated_warmuped_ft \
  --ancphore_path /content/DiffPhore/programs/ \
  --cache_path /content/caches --out_dir /content/out \
  --sample_per_complex 4 --batch_size 4 --num_workers 2

## 5. Check the output

A successful run writes `ranked_results.csv` and a ranked pose file. The repository ships
its own reference output for this same example in `examples/output/1/`, so we can put the
two side by side.

The fitness scores will not match exactly. Sampling is stochastic and we asked for 4 poses
instead of 40. What matters is that a score comes out at all and is in a similar range.

In [ ]:
import pandas as pd, pathlib
ours = pathlib.Path("/content/out/ranked_results.csv")
ref  = pathlib.Path("/content/DiffPhore/examples/output/1/ranked_results.csv")
print("Ran OK:", ours.exists())
if ours.exists():
    display(pd.read_csv(ours))
print("Reference shipped with the repo:")
display(pd.read_csv(ref))

## 6. Look at a pose

If the run worked there is an SDF of aligned poses. Count the molecules in it as a final
check that the output is real rather than an empty file.

In [ ]:
from rdkit import Chem
import glob
poses = glob.glob("/content/out/ranked_poses/*.sdf")
print("Pose files:", poses)
if poses:
    mols = [m for m in Chem.SDMolSupplier(poses[0]) if m is not None]
    print(f"{len(mols)} poses read from {poses[0]}")
    print(Chem.MolToSmiles(mols[0]) if mols else "no valid molecules")

## Summary

- If section 4 completed and section 5 printed a fitness score, DiffPhore works in Colab
  and we can think about what to point it at.
- If it failed, the error almost certainly comes from section 3: the 2022-era graph stack
  against current PyTorch. Copy the traceback back into the chat.
- Known rough edges: the README names `environment.yml` but the file is
  `src/environment_diffphore.yml`, and `--ancphore_path` is relative by default.

**Next:** decide whether DiffPhore earns a place in a project, or whether RDKit's own
pharmacophore features are enough.